# 현실의 데이터 14: 네트워크 침입 탐지

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/14_cyber_security.ipynb)

**분야**: 플랫폼 · 서비스 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 보안관제 · 침입 탐지 (IDS) · 처음 보는 공격

> 네트워크 연결 15만 건. 처음 보는 공격도 잡아낼 수 있을까?

**데이터**: NSL-KDD (약 22MB). Tavallaee et al. (2009). KDD Cup 99 를 정리한 침입 탐지 데이터: 연결 한 건마다 41개 숫자 (길이 · 프로토콜 · 서비스 · 바이트 수 · 오류 비율 …) + 정상 / 공격 이름. 학습 125,973건 · 시험 22,544건 (시험에만 있는 공격 포함).

산업 현장의 문제와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 네트워크 연결 한 건 |
| ② 입력 X | 41개 숫자 · 범주 (프로토콜 · 서비스 · 바이트 수 · 실패 비율 …) |
| ③ 출력 y | 정상 / 공격 |
| ④ 평가지표 | 재현율 (놓친 공격) · 정밀도 (헛경보) |
| ⑤ 기준선 | 전부 '정상' |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Tavallaee et al. (2009) A detailed analysis of the KDD CUP 99 data set, IEEE CISDA (NSL-KDD)](https://doi.org/10.1109/CISDA.2009.5356528)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
COLS = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
        "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
        "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
        "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
        "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
        "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "attack", "difficulty"]
BASE = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
train = pd.read_csv(download(BASE + "KDDTrain+.txt", "data/nslkdd_train.txt"), header=None, names=COLS)
test = pd.read_csv(download(BASE + "KDDTest+.txt", "data/nslkdd_test.txt"), header=None, names=COLS)
print(train.shape, test.shape)
train.sample(5, random_state=0).iloc[:, [0, 1, 2, 3, 4, 5, 22, 41]]

(125973, 43) (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,count,attack
11224,0,tcp,http,SF,206,3581,1,normal
13951,0,tcp,http,SF,239,2239,5,normal
62642,0,tcp,smtp,SF,1271,442,1,normal
78373,0,tcp,courier,REJ,0,0,121,neptune
22304,0,udp,private,SF,1,0,22,satan


## 열어 보기

공격 종류와 개수. 그리고 **시험에만 있는 공격** 은 몇 가지일까?

In [3]:
print("train:", train["attack"].value_counts().head(8).to_dict())
new_attacks = sorted(set(test["attack"]) - set(train["attack"]))
print(f"attack types only in test: {len(new_attacks)} ->", new_attacks[:10])
print(f"test rows with a new attack type: {test['attack'].isin(new_attacks).mean():.1%}")
train["y"] = (train["attack"] != "normal").astype(int); test["y"] = (test["attack"] != "normal").astype(int)
print(f"attack share: train {train['y'].mean():.1%}, test {test['y'].mean():.1%}")

train: {'normal': 67343, 'neptune': 41214, 'satan': 3633, 'ipsweep': 3599, 'portsweep': 2931, 'smurf': 2646, 'nmap': 1493, 'back': 956}
attack types only in test: 17 -> ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack']
test rows with a new attack type: 16.6%
attack share: train 46.5%, test 56.9%


## 문제 하나 골라 보기: 연결 → 공격일까

같은 모델 (랜덤 포레스트) 을 두 가지로 채점한다.
1. 학습 데이터를 무작위로 나눈 시험 (같은 공격들)
2. 공식 시험 데이터 (처음 보는 공격 포함)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def precision_recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1)); fp = np.sum((y_pred == 1) & (y_true == 0)); fn = np.sum((y_pred == 0) & (y_true == 1))
    return (tp / (tp + fp) if tp + fp else 0.0), tp / (tp + fn)

both = pd.get_dummies(pd.concat([train, test])[COLS[:-2]], columns=["protocol_type", "service", "flag"], dtype=float)
Xtr_all, Xte_off = both.iloc[:len(train)], both.iloc[len(train):]
Xa, Xb, ya, yb = train_test_split(Xtr_all, train["y"].values, test_size=0.3, random_state=0)
m = RandomForestClassifier(200, random_state=0, n_jobs=-1).fit(Xa, ya)
for name, X, y in [("random split of train", Xb, yb), ("official test (new attacks)", Xte_off, test["y"].values)]:
    pred = m.predict(X); p, r = precision_recall(y, pred)
    print(f"{name:28s} accuracy {np.mean(pred == y):.1%}  precision {p:.1%}  recall {r:.1%}")
pred = m.predict(Xte_off); new = test["attack"].isin(new_attacks).values
print(f"recall on attack types never seen in training: {np.mean(pred[new] == 1):.1%}")

random split of train        accuracy 99.9%  precision 99.9%  recall 99.8%
official test (new attacks)  accuracy 77.0%  precision 96.8%  recall 61.6%
recall on attack types never seen in training: 25.2%


**관찰**: 같은 공격끼리 나눈 시험에서는 거의 100% 이지만, 처음 보는 공격이 섞이면 공격을 많이 놓친다. 보안에서는 **공격자가 계속 바뀐다**. 시험 데이터를 '미래의 공격' 처럼 만들어야 진짜 점수가 나온다.

## 다른 문제로 바꿔 보기

- **이상 탐지**: 공격 라벨 없이 정상 연결만 배워서 '평소와 다른 연결' 을 찾으면? (4주차 가우시안 · k-means)
- 보안관제 요원이 하루에 볼 수 있는 경보가 100개뿐이라면? 지표를 '상위 100개 중 진짜 공격 수' 로 바꾼다.

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |